# 🔬 Vision Privacy & Identity Lab – Dreambooth / LoRA Training

This notebook fine-tunes a Stable Diffusion model on your personal portrait dataset using **Dreambooth + LoRA** inside Google Colab.

### Quick-start
1. Upload your preprocessed dataset to Google Drive (or run `preprocessing.py` in the cell below).
2. Fill in the configuration variables in **§ 2 – Configuration**.
3. Run all cells top-to-bottom (`Runtime → Run all`).

> **VRAM tips** – this notebook enables:
> - `gradient_checkpointing` (trades compute for memory)
> - `xformers` memory-efficient attention (when available)
> - 8-bit Adam via `bitsandbytes` (halves optimiser memory)
> - mixed-precision `fp16` training

## § 1 – Environment Setup

In [ ]:
# ── Check GPU ────────────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Install / upgrade dependencies ───────────────────────────────────────────
!pip install -q --upgrade \
    diffusers==0.21.4 \
    transformers==4.33.3 \
    accelerate==0.23.0 \
    safetensors \
    bitsandbytes \
    mediapipe \
    Pillow \
    opencv-python-headless \
    xformers

# Install the official Dreambooth training script from diffusers examples
!pip install -q git+https://github.com/huggingface/peft.git

print("✅ Dependencies installed")

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted at /content/drive")

## § 2 – Configuration
**Edit these variables before running training.**

In [ ]:
import os

# ── Subject / token ──────────────────────────────────────────────────────────
UNIQUE_TOKEN   = "sks"          # rare token that represents YOUR subject
CLASS_NOUN     = "person"       # general class noun  (e.g. person, woman, man)
INSTANCE_PROMPT = f"a photo of {UNIQUE_TOKEN} {CLASS_NOUN}"
CLASS_PROMPT    = f"a photo of {CLASS_NOUN}"

# ── Paths (Google Drive) ─────────────────────────────────────────────────────
DRIVE_ROOT          = "/content/drive/MyDrive/VisionPrivacyLab"
INSTANCE_DATA_DIR   = f"{DRIVE_ROOT}/dataset"           # preprocessed images + captions
CLASS_DATA_DIR      = f"{DRIVE_ROOT}/class_images"      # prior-preservation class images
OUTPUT_DIR          = f"{DRIVE_ROOT}/output_model"      # saved LoRA weights

# ── Base model ───────────────────────────────────────────────────────────────
PRETRAINED_MODEL    = "runwayml/stable-diffusion-v1-5"  # or a local path

# ── Training hyper-parameters ────────────────────────────────────────────────
RESOLUTION          = 512
TRAIN_BATCH_SIZE    = 1
GRADIENT_ACCUM_STEPS = 4
NUM_TRAIN_STEPS     = 800
LEARNING_RATE       = 1e-4
LR_SCHEDULER        = "constant"
LR_WARMUP_STEPS     = 0
MAX_GRAD_NORM       = 1.0

# ── Prior-preservation ───────────────────────────────────────────────────────
WITH_PRIOR          = True    # set False to disable prior-preservation loss
PRIOR_LOSS_WEIGHT   = 1.0
NUM_CLASS_IMAGES    = 200

# ── LoRA ─────────────────────────────────────────────────────────────────────
USE_LORA            = True
LORA_RANK           = 4

# ── Memory-saving ────────────────────────────────────────────────────────────
USE_GRADIENT_CHECKPOINTING = True
USE_8BIT_ADAM              = True
MIXED_PRECISION            = "fp16"   # "bf16" on A100/H100

# ── Checkpointing ────────────────────────────────────────────────────────────
SAVE_STEPS          = 200

# Ensure output directories exist
os.makedirs(INSTANCE_DATA_DIR, exist_ok=True)
os.makedirs(CLASS_DATA_DIR,    exist_ok=True)
os.makedirs(OUTPUT_DIR,        exist_ok=True)

print(f"Instance prompt : {INSTANCE_PROMPT}")
print(f"Class prompt    : {CLASS_PROMPT}")
print(f"Output dir      : {OUTPUT_DIR}")

## § 3 – (Optional) Run Preprocessing
Skip this section if you already have a preprocessed dataset in `INSTANCE_DATA_DIR`.

In [ ]:
RAW_IMAGES_DIR = f"{DRIVE_ROOT}/raw_images"   # ← put your raw photos here

# Download preprocessing script from repo
!wget -q https://raw.githubusercontent.com/ap-xlr8/IA-/main/preprocessing.py -O preprocessing.py

!python preprocessing.py \
    --input_dir  "{RAW_IMAGES_DIR}" \
    --output_dir "{INSTANCE_DATA_DIR}" \
    --token      "{UNIQUE_TOKEN} {CLASS_NOUN}" \
    --size       {RESOLUTION} \
    --smooth_skin
    # add --inpaint_tattoos if needed (requires ~10 GB VRAM)

print("✅ Preprocessing complete")

## § 4 – Training

In [ ]:
# ── Download the official DreamBooth+LoRA training script ────────────────────
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora.py \
    -O train_dreambooth_lora.py
print("✅ Training script downloaded")

In [ ]:
import subprocess, shlex

# Build command-line arguments
cmd_parts = [
    "accelerate launch train_dreambooth_lora.py",
    f"--pretrained_model_name_or_path='{PRETRAINED_MODEL}'",
    f"--instance_data_dir='{INSTANCE_DATA_DIR}'",
    f"--output_dir='{OUTPUT_DIR}'",
    f"--instance_prompt='{INSTANCE_PROMPT}'",
    f"--resolution={RESOLUTION}",
    f"--train_batch_size={TRAIN_BATCH_SIZE}",
    f"--gradient_accumulation_steps={GRADIENT_ACCUM_STEPS}",
    f"--learning_rate={LEARNING_RATE}",
    f"--lr_scheduler={LR_SCHEDULER}",
    f"--lr_warmup_steps={LR_WARMUP_STEPS}",
    f"--max_train_steps={NUM_TRAIN_STEPS}",
    f"--max_grad_norm={MAX_GRAD_NORM}",
    f"--mixed_precision={MIXED_PRECISION}",
    f"--checkpointing_steps={SAVE_STEPS}",
    f"--lora_rank={LORA_RANK}",
]

if USE_GRADIENT_CHECKPOINTING:
    cmd_parts.append("--gradient_checkpointing")

if USE_8BIT_ADAM:
    cmd_parts.append("--use_8bit_adam")

if WITH_PRIOR:
    cmd_parts += [
        f"--with_prior_preservation",
        f"--prior_loss_weight={PRIOR_LOSS_WEIGHT}",
        f"--class_data_dir='{CLASS_DATA_DIR}'",
        f"--class_prompt='{CLASS_PROMPT}'",
        f"--num_class_images={NUM_CLASS_IMAGES}",
    ]

full_cmd = " \\
    ".join(cmd_parts)
print("Running:\n", full_cmd)
!{full_cmd}

## § 5 – Inference / Image Generation

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

# ── Load base model + LoRA weights ───────────────────────────────────────────
pipe = StableDiffusionPipeline.from_pretrained(
    PRETRAINED_MODEL,
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

pipe.load_lora_weights(OUTPUT_DIR)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers enabled")
except Exception:
    print("xformers not available – using default attention")

print("✅ Model loaded")

In [ ]:
# ── Generate images ───────────────────────────────────────────────────────────
INFERENCE_PROMPTS = [
    f"a professional headshot of {UNIQUE_TOKEN} {CLASS_NOUN}, studio lighting, 4k",
    f"{UNIQUE_TOKEN} {CLASS_NOUN} smiling, natural light, sharp focus",
    f"oil painting portrait of {UNIQUE_TOKEN} {CLASS_NOUN}",
]

NEGATIVE_PROMPT  = (
    "blurry, distorted face, extra limbs, deformed, low quality, "
    "text, watermark, signature"
)
NUM_IMAGES       = 4
GUIDANCE_SCALE   = 7.5
NUM_STEPS        = 30
SEED             = 42

generator = torch.Generator(device="cuda").manual_seed(SEED)

for prompt in INFERENCE_PROMPTS:
    print(f"\nPrompt: {prompt}")
    images = pipe(
        prompt          = prompt,
        negative_prompt = NEGATIVE_PROMPT,
        num_images_per_prompt = NUM_IMAGES,
        num_inference_steps   = NUM_STEPS,
        guidance_scale        = GUIDANCE_SCALE,
        generator             = generator,
    ).images

    for i, img in enumerate(images):
        display(img)
        out_path = f"{OUTPUT_DIR}/inference_{prompt[:30].replace(' ','_')}_{i}.png"
        img.save(out_path)
        print(f"  Saved → {out_path}")

---
## § 6 – Save LoRA Weights to Drive
Your fine-tuned LoRA weights are already saved in `OUTPUT_DIR` on Google Drive after training completes.
Use the cell below to create a timestamped ZIP archive for easier portability.

In [ ]:
import shutil, datetime

timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive    = f"{DRIVE_ROOT}/lora_weights_{UNIQUE_TOKEN}_{timestamp}"
shutil.make_archive(archive, "zip", OUTPUT_DIR)
print(f"✅ Archive saved → {archive}.zip")